# Data Ingestion
First, we will read in the Reddit data from JSON files.

## Plan
The goal is to create a database that contains these tables:

1. Comments/Replies (based on JSON files)
2. Submissions (based on Replies)
3. Users (based on Replies and Submissions)

In [1]:
import pandas as pd
import duckdb
from pathlib import Path

In [2]:
# SET PATHS
data_dir   = Path("../data/LoveIslandUSA_S8/")
json_files = list(data_dir.glob("*.json"))

print(
    f"N JSON FILES: {len(json_files)}"
)

N JSON FILES: 58


## Comments/Replies

In [3]:
# 1. Read all JSON files into a list of dictionaries/DataFrames
# We use a list comprehension to process each Path object
all_replies = []
for file_path in json_files:
    # pd.read_json reads the file into a DataFrame
    temp_df = pd.read_json(file_path)
    
    # Optional: Keep track of the filename in the data
    # This is helpful for debugging later if you find a data issue
    temp_df['source_filename'] = file_path.name
    
    all_replies.append(temp_df)

# 2. Bind the rows (concatenate into one large DataFrame)
# 'ignore_index=True' gives you a clean 0..N index
all_replies_df = pd.concat(all_replies, ignore_index=True)

In [4]:
# Check the results
print(all_replies_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4320 entries, 0 to 4319
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   title            4320 non-null   object 
 1   name             4320 non-null   object 
 2   url              4320 non-null   object 
 3   selftext         4320 non-null   object 
 4   score            4320 non-null   int64  
 5   upvote_ratio     4320 non-null   float64
 6   permalink        4320 non-null   object 
 7   id               4320 non-null   object 
 8   author           4320 non-null   object 
 9   link_flair_text  4320 non-null   object 
 10  num_comments     4320 non-null   int64  
 11  over_18          4320 non-null   bool   
 12  spoiler          4320 non-null   bool   
 13  pinned           4320 non-null   bool   
 14  locked           4320 non-null   bool   
 15  distinguished    2856 non-null   object 
 16  created_utc      4320 non-null   int64  
 17  comments      

In [5]:
all_replies_df.head(3)

,title,name,url,selftext,score,upvote_ratio,permalink,id,author,link_flair_text,num_comments,over_18,spoiler,pinned,locked,distinguished,created_utc,comments,source_filename
0,Season 8 - Episode 7 - Cast Opinions Discussion,t3_1u1o6bb,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's discuss!\n\nShare your opinions on the...,14,0.85,/r/LoveIslandUSA/comments/1u1o6bb/season_8_epi...,1u1o6bb,AutoModerator,✴️ OPINIONS MEGATHREAD ✴️,374,False,False,False,False,NaN,1781055028,"{'author': 'AutoModerator', 'id': 'oqr6sxe', '...",AutoModerator_Season 8 - Episode 7 - Cast Opin...
1,Season 8 - Episode 7 - Cast Opinions Discussion,t3_1u1o6bb,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's discuss!\n\nShare your opinions on the...,14,0.85,/r/LoveIslandUSA/comments/1u1o6bb/season_8_epi...,1u1o6bb,AutoModerator,✴️ OPINIONS MEGATHREAD ✴️,374,False,False,False,False,NaN,1781055028,"{'author': 'Lazy_Chemistry_3899', 'id': 'oqrjr...",AutoModerator_Season 8 - Episode 7 - Cast Opin...
2,Season 8 - Episode 7 - Cast Opinions Discussion,t3_1u1o6bb,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's discuss!\n\nShare your opinions on the...,14,0.85,/r/LoveIslandUSA/comments/1u1o6bb/season_8_epi...,1u1o6bb,AutoModerator,✴️ OPINIONS MEGATHREAD ✴️,374,False,False,False,False,NaN,1781055028,"{'author': 'daemonxcaraxes', 'id': 'oqrhf82', ...",AutoModerator_Season 8 - Episode 7 - Cast Opin...


We now want to unpack the nesting.

In [6]:
def flatten_full_thread(comment_node):
    """Flattens a comment tree including the root node."""
    all_rows = []
    
    # We use a helper to traverse depth-first
    def _traverse(node, parent_id, depth):
        # Create record for this node
        record = {
            **node,
            'parent_id': parent_id,
            'depth': depth,
            'replied_to_top': (depth == 2), # If depth is 2, it's a direct reply to root
            'replies': None
        }
        all_rows.append(record)
        
        # Recurse through replies
        for reply in node.get('replies', []):
            _traverse(reply, node.get('id'), depth + 1)
            
    # Start traversal at depth 1 (the root)
    _traverse(comment_node, comment_node.get('parent_id'), 1)
    return pd.DataFrame(all_rows)

I am testing on one file.

In [7]:
this_json = pd.read_json(json_files[0])

# 2. Main processing loop to attach metadata
master_list = []

for _, row in this_json.iterrows():
    # Flatten the root and all its descendants
    thread_df = flatten_full_thread(row['comments'])
    
    # Attach post-level metadata to every comment row
    thread_df['post_title'] = row['title']
    thread_df['post_id'] = row['id']
    thread_df['post_score'] = row['score']
    
    master_list.append(thread_df)

# 3. Create the final "Tidy" DataFrame
final_flat_df = pd.concat(master_list, ignore_index=True)

In [8]:
final_flat_df.head(3)

,author,id,score,subreddit,author_flair,submission,stickied,body,is_submitter,distinguished,created_utc,parent_id,replies,depth,replied_to_top,post_title,post_id,post_score
0,AutoModerator,oqr6sxe,1,LoveIslandUSA,None,1u1o6bb,True,### **SORT** this Discussion &#x1F3A8;\n---\n-...,True,moderator,1.781055e+09,t3_1u1o6bb,None,1,False,Season 8 - Episode 7 - Cast Opinions Discussion,1u1o6bb,14
1,Lazy_Chemistry_3899,oqrjrnv,153,LoveIslandUSA,None,1u1o6bb,False,When Bryce bragged to the guys how kind and ni...,False,None,1.781059e+09,t3_1u1o6bb,None,1,False,Season 8 - Episode 7 - Cast Opinions Discussion,1u1o6bb,14
2,katieofgilead,oqrmj7f,35,LoveIslandUSA,None,1u1o6bb,False,Loooved that! He is looking way past her beaut...,False,None,1.781060e+09,oqrjrnv,None,2,True,Season 8 - Episode 7 - Cast Opinions Discussion,1u1o6bb,14


In [9]:
# This will hold the result for every post from every file
master_list = []

# 2. Outer loop: Iterate through every file
for file_path in json_files:
    # Load the specific JSON file
    # (Note: pd.read_json is fine if the file structure matches your previous this_json)
    df_current_file = pd.read_json(file_path)
    
    # 3. Inner loop: Iterate through every post in that file
    for _, row in df_current_file.iterrows():
        # Ensure we have a valid dictionary to flatten
        comment_root = row['comments']
        if not isinstance(comment_root, dict):
            continue
            
        # Flatten the root and all its descendants
        thread_df = flatten_full_thread(comment_root)
        
        # Attach post-level metadata
        thread_df['post_title'] = row['title']
        thread_df['post_id'] = row['id']
        thread_df['post_score'] = row['score']
        thread_df['source_file'] = file_path # Helpful for tracing
        
        master_list.append(thread_df)

In [10]:
# 3. Create the final Tidy DataFrame
final_master_df = pd.concat(master_list, ignore_index=True)

print(f"Flattened {len(final_master_df)} total comments across all files.")

Flattened 11458 total comments across all files.


In [11]:
final_master_df.head(3)

,author,id,score,subreddit,author_flair,submission,stickied,body,is_submitter,distinguished,created_utc,parent_id,replies,depth,replied_to_top,post_title,post_id,post_score,source_file
0,AutoModerator,oqr6sxe,1,LoveIslandUSA,None,1u1o6bb,True,### **SORT** this Discussion &#x1F3A8;\n---\n-...,True,moderator,1.781055e+09,t3_1u1o6bb,None,1,False,Season 8 - Episode 7 - Cast Opinions Discussion,1u1o6bb,14,../data/LoveIslandUSA_S8/AutoModerator_Season ...
1,Lazy_Chemistry_3899,oqrjrnv,153,LoveIslandUSA,None,1u1o6bb,False,When Bryce bragged to the guys how kind and ni...,False,None,1.781059e+09,t3_1u1o6bb,None,1,False,Season 8 - Episode 7 - Cast Opinions Discussion,1u1o6bb,14,../data/LoveIslandUSA_S8/AutoModerator_Season ...
2,katieofgilead,oqrmj7f,35,LoveIslandUSA,None,1u1o6bb,False,Loooved that! He is looking way past her beaut...,False,None,1.781060e+09,oqrjrnv,None,2,True,Season 8 - Episode 7 - Cast Opinions Discussion,1u1o6bb,14,../data/LoveIslandUSA_S8/AutoModerator_Season ...


In [12]:
# 1. Connect to the existing database
con = duckdb.connect("../data/love_island_reddit.duckdb")

# 2. Write the comments DataFrame to a new table
# Using 'replace' if you want to overwrite, or 'create' if it's new
con.execute("CREATE OR REPLACE TABLE comments AS SELECT * FROM final_master_df")

# 3. Create an index or optimize
# DuckDB doesn't use indexes the same way as Postgres, 
# but you can ensure your IDs are query-ready
con.execute("CREATE INDEX idx_comment_id ON comments(id)")
con.execute("CREATE INDEX idx_parent_id ON comments(parent_id)")

# 4. Timestamp
con.execute("ALTER TABLE comments ADD COLUMN created_at TIMESTAMP")
con.execute("UPDATE comments SET created_at = to_timestamp(created_utc)")

In [13]:
# 4. Select a sample to see if the timestamp looks correct
sample = con.execute("SELECT created_utc, created_at FROM comments LIMIT 5").df()
print(sample)

con.close()

    created_utc          created_at
0  1.781055e+09 2026-06-09 18:30:28
1  1.781059e+09 2026-06-09 19:44:08
2  1.781060e+09 2026-06-09 20:00:32
3  1.781062e+09 2026-06-09 20:31:02
4  1.781064e+09 2026-06-09 20:56:13


## Submissions (From JSON)

In [14]:
this_json = pd.read_json(json_files[0])

# 1. Drop the 'comments' column
# 2. Apply distinct (drop_duplicates)
submissions_df = this_json.drop(columns=['comments']).drop_duplicates()

In [15]:
submissions_df.head(3)

,title,name,url,selftext,score,upvote_ratio,permalink,id,author,link_flair_text,num_comments,over_18,spoiler,pinned,locked,distinguished,created_utc
0,Season 8 - Episode 7 - Cast Opinions Discussion,t3_1u1o6bb,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's discuss!\n\nShare your opinions on the...,14,0.85,/r/LoveIslandUSA/comments/1u1o6bb/season_8_epi...,1u1o6bb,AutoModerator,✴️ OPINIONS MEGATHREAD ✴️,374,False,False,False,False,NaN,1781055028


In [16]:
all_submissions = []

for this_json in json_files:
    # Read the JSON file into a DataFrame
    temp_df = pd.read_json(this_json)
    
    # Drop the 'comments' column and apply distinct
    temp_df = temp_df.drop(columns=['comments']).drop_duplicates()
    
    # Append to the list of all submissions
    all_submissions.append(temp_df)

In [17]:
all_submissions_df = pd.concat(all_submissions, ignore_index=True)

In [18]:
all_submissions_df.head(3)

,title,name,url,selftext,score,upvote_ratio,permalink,id,author,link_flair_text,num_comments,over_18,spoiler,pinned,locked,distinguished,created_utc
0,Season 8 - Episode 7 - Cast Opinions Discussion,t3_1u1o6bb,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's discuss!\n\nShare your opinions on the...,14,0.85,/r/LoveIslandUSA/comments/1u1o6bb/season_8_epi...,1u1o6bb,AutoModerator,✴️ OPINIONS MEGATHREAD ✴️,374,False,False,False,False,NaN,1781055028
1,[WITH Ads] Season 8 - Episode 10 -| 9 PM EST,t3_1u570dj,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's chat!\n\nThe with-ads live viewing sta...,6,0.88,/r/LoveIslandUSA/comments/1u570dj/with_ads_sea...,1u570dj,schedulerplus,LIVE DISCUSSION,55,False,False,False,False,moderator,1781398145
2,[NONSPOILER] Season 8 - Episode 32 - Thursday ...,t3_1us8brl,https://www.reddit.com/r/LoveIslandUSA/comment...,# Let's chat!\n\nThe live discussion **spoiler...,9,0.84,/r/LoveIslandUSA/comments/1us8brl/nonspoiler_s...,1us8brl,schedulerplus,LIVE DISCUSSION | 😇 NONSPOILER 😇,220,False,False,False,False,moderator,1783643645


In [19]:
# Check the results
print(all_submissions_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   title            58 non-null     object 
 1   name             58 non-null     object 
 2   url              58 non-null     object 
 3   selftext         58 non-null     object 
 4   score            58 non-null     int64  
 5   upvote_ratio     58 non-null     float64
 6   permalink        58 non-null     object 
 7   id               58 non-null     object 
 8   author           58 non-null     object 
 9   link_flair_text  58 non-null     object 
 10  num_comments     58 non-null     int64  
 11  over_18          58 non-null     bool   
 12  spoiler          58 non-null     bool   
 13  pinned           58 non-null     bool   
 14  locked           58 non-null     bool   
 15  distinguished    40 non-null     object 
 16  created_utc      58 non-null     int64  
dtypes: bool(4), float6

In [20]:
# Connect to a database file (this creates it if it doesn't exist)
con = duckdb.connect("../data/love_island_reddit.duckdb")

# 3. Create a table in DuckDB directly from the Pandas DataFrame
con.execute("CREATE OR REPLACE TABLE submissions AS SELECT * FROM all_submissions_df")

con.execute("ALTER TABLE submissions ADD COLUMN created_at TIMESTAMP")
con.execute("UPDATE submissions SET created_at = to_timestamp(created_utc)")

# Close it up
con.close()

## Users

In [21]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

In [22]:
# 1. Check count of authors in each table independently
print(con.execute("SELECT count(DISTINCT author) FROM submissions").fetchone())
print(con.execute("SELECT count(DISTINCT author) FROM comments").fetchone())

# 2. Check for null authors
print(con.execute("SELECT count(*) FROM comments WHERE author IS NULL").fetchone())

(3,)
(2903,)
(0,)


In [23]:
# 1. To see every single username (including duplicates for repeat posters):
all_names = con.execute("SELECT author FROM submissions").df()

# 2. To see only unique usernames (often better for identifying the list of contributors):
unique_names = con.execute("SELECT DISTINCT author FROM submissions WHERE author IS NOT NULL").df()

print("List of unique authors who created submissions:")
print(unique_names)

List of unique authors who created submissions:
          author
0    Ok_Side7135
1  AutoModerator
2  schedulerplus


In [24]:
# 1. Create the users table by pulling unique authors from both sources
con.execute("""
    CREATE OR REPLACE TABLE users AS 
    SELECT DISTINCT author 
    FROM (
        SELECT author FROM submissions
        UNION
        SELECT author FROM comments
    ) 
    WHERE author IS NOT NULL
""")

# 2. Add the 'bot' column
con.execute("ALTER TABLE users ADD COLUMN bot VARCHAR DEFAULT 'N'")

# 3. Logic to flag bots and moderators
# Rule A: Flag common bot keywords (like 'bot' or 'automoderator')
con.execute("UPDATE users SET bot = 'Y' WHERE author ILIKE '%bot%' OR author ILIKE '%automoderator%'")

# Rule B: Flag moderators (those marked 'distinguished' in comments)
# This uses a subquery to update all mods at once (much faster than a loop)
con.execute("""
    UPDATE users 
    SET bot = 'Y' 
    WHERE author IN (
        SELECT DISTINCT author 
        FROM comments 
        WHERE distinguished IS NOT NULL
    )
""")

# 4. Final count check
counts = con.execute("SELECT bot, count(*) FROM users GROUP BY bot").fetchall()
print(f"User classification counts: {counts}")

User classification counts: [('N', 2895), ('Y', 10)]


In [25]:
# 1. Add the new columns
con.execute("ALTER TABLE users ADD COLUMN IF NOT EXISTS num_submissions INTEGER DEFAULT 0")
con.execute("ALTER TABLE users ADD COLUMN IF NOT EXISTS num_comments INTEGER DEFAULT 0")

# 2. Update with submission counts
con.execute("""
    UPDATE users 
    SET num_submissions = sub_counts.total
    FROM (
        SELECT author, COUNT(*) as total 
        FROM submissions 
        GROUP BY author
    ) as sub_counts
    WHERE users.author = sub_counts.author
""")

# 3. Update with comment counts
con.execute("""
    UPDATE users 
    SET num_comments = com_counts.total
    FROM (
        SELECT author, COUNT(*) as total 
        FROM comments 
        GROUP BY author
    ) as com_counts
    WHERE users.author = com_counts.author
""")

In [26]:
# 4. Verify the user profiles
profile_sample = con.execute("SELECT * FROM users WHERE num_comments > 5 OR num_submissions > 0 LIMIT 10").df()
print(profile_sample)

                author bot  num_submissions  num_comments
0              DELETED   N                0           243
1              jmood88   N                0            20
2  dependable_mushroom   N                0            75
3             nymaeris   N                0            15
4         owlbethchase   N                0            13
5    mossbetweenmytoes   N                0             6
6        Starwhisperer   N                0            23
7                wupao   N                0            15
8     cinderellahottie   N                0            12
9                 fp_6   N                0            14


In [27]:
# Close connection
con.close()

# Counts

In [28]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

# Use .fetchone() to get the count value directly
result = con.execute("SELECT DISTINCT COUNT(*) FROM submissions").fetchone()

print(f"Total rows in submissions: {result[0]}")

con.close()

Total rows in submissions: 58


In [29]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

# Use .fetchone() to get the count value directly
result = con.execute("SELECT DISTINCT COUNT(*) FROM comments").fetchone()

print(f"Total rows in comments: {result[0]}")

con.close()

Total rows in comments: 11458


In [30]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

# Use .fetchone() to get the count value directly
result = con.execute("SELECT DISTINCT COUNT(*) FROM users").fetchone()

print(f"Total rows in users: {result[0]}")

con.close()

Total rows in users: 2905


# Cleaning

## Remove [deleted]

In [31]:
# 1. Connect to the database
con = duckdb.connect("../data/love_island_reddit.duckdb")

# 2. Verify how many will be deleted (sanity check)
result = con.execute("SELECT count(*) FROM comments WHERE body IN ('[deleted]', '[removed]');").fetchone()
print(f"Deleting {result[0]} rows...")

# 3. Perform the delete operation
con.execute("DELETE FROM comments WHERE body IN ('[deleted]', '[removed]')")

# 4. (Optional but recommended) Optimize the database
# Deleting rows leaves 'holes' in the file. Vacuuming shrinks the file size.
con.execute("VACUUM ANALYZE")

# 5. Verify the count is now zero
final_check = con.execute("SELECT count(*) FROM comments WHERE body IN ('[deleted]', '[removed]');").fetchone()
print(f"Remaining deleted/removed comments: {final_check[0]}")

con.close()

Deleting 238 rows...
Remaining deleted/removed comments: 0


In [32]:
import duckdb
con = duckdb.connect("../data/love_island_reddit.duckdb")

# This shows the columns and types for your tables
print(con.execute("DESCRIBE submissions").df())
print(con.execute("DESCRIBE comments").df())
print(con.execute("DESCRIBE users").df())

con.close()

        column_name column_type null   key default extra
0             title     VARCHAR  YES  None    None  None
1              name     VARCHAR  YES  None    None  None
2               url     VARCHAR  YES  None    None  None
3          selftext     VARCHAR  YES  None    None  None
4             score      BIGINT  YES  None    None  None
5      upvote_ratio      DOUBLE  YES  None    None  None
6         permalink     VARCHAR  YES  None    None  None
7                id     VARCHAR  YES  None    None  None
8            author     VARCHAR  YES  None    None  None
9   link_flair_text     VARCHAR  YES  None    None  None
10     num_comments      BIGINT  YES  None    None  None
11          over_18     BOOLEAN  YES  None    None  None
12          spoiler     BOOLEAN  YES  None    None  None
13           pinned     BOOLEAN  YES  None    None  None
14           locked     BOOLEAN  YES  None    None  None
15    distinguished     VARCHAR  YES  None    None  None
16      created_utc      BIGINT